# T8 It3: a two-tier rule built on NUDT, checked on official TRY, then pre-registered

**Idea** (block `train/checks/two_tier.py`, reasons in its docstring): at 5-11 inliers the wrong choices look the same on official and NUDT
(DINO z ~1.2, a stronger DINO place elsewhere in the query), while right ones have z >= ~2 and are DINO's top place of their query. So:
*tier 1* answer at >= i_hi inliers (as today); *tier 2* answer at >= a_low inliers only if z >= z0 and zgap >= g0. No learned model, 3-4 numbers.

**Protocol:**
1. Grid on the **five NUDT sets only** (design). Keep rules with 0 wrong answers on every NUDT set. Pick the best mean NUDT score; ties go to the
   stricter rule (higher z0, then g0, then a_low). This order was fixed before the grid. (NUDT 606 was first kept aside as a check; the Jetson
   rule chosen on 101-505 made 2 wrong answers there, the same as the Jetson today, so 606 became design data and the selection was redone on
   all five sets.)
2. Official **TRY** only checks the chosen rule (and the Jetson early-exit level); official TEST is not loaded.
3. Everything that will be scored is written to `results/preregistered_t8.json` with its time, before It4 opens official TEST and the blind
   NUDT 707 draw.

In [1]:
import os, sys, time, json
os.environ.setdefault("OMP_NUM_THREADS", str(len(os.sched_getaffinity(0))))
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import numpy as np, pandas as pd, joblib
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
from train.checks.candidate_judge import (FEATURES_CTX, TAUS, baseline, decide, judge_new, judge_probability, ltt_tau, scan_tau, summary)
from train.checks.holdout import NUDT_DESIGN, answer_change, bar_frontier, load_t8_table, versus_frontier
from train.checks.two_tier import null_decide, null_pvalues
RES = ROOT / "tests/Test_Iteration_8/results"
table = load_t8_table(ROOT, locked=True)          # official_test is not loaded here
SEEN = ["official_try"] + NUDT_DESIGN
assert "official_test" not in set(table.data)
table.groupby("data").photo.nunique()
import itertools, datetime, hashlib
from joblib import Parallel, delayed
from train.checks.two_tier import GRID, decide_two_tier
from train.checks.holdout import attitude_frame, with_attitude
from train.checks import pose_bias

## 1. Grid on the NUDT sets

In [2]:
nudt = table[table.data.isin(NUDT_DESIGN)].reset_index(drop=True)
grid = [g for g in itertools.product(*GRID.values()) if g[0] < g[3]]
def one(g):
    rule = dict(zip(GRID.keys(), g)); out = []
    for var in ["pc", "jetson"]:
        s = summary(decide_two_tier(nudt, rule, var))
        out.append(dict(var=var, **rule, s_design=s.score.mean(), w_design=int(s.wrong.sum()), **{f"s_{d}": s.loc[d, "score"] for d in s.index}))
    return out
t0 = time.time()
G = pd.DataFrame([r for rr in Parallel(n_jobs=min(10, len(os.sched_getaffinity(0))))(delayed(one)(g) for g in grid) for r in rr])
G.to_csv(RES / "it3_grid_nudt.csv", index=False); print(len(grid), "rules, %.0f s" % (time.time() - t0))
CHOSEN = {}
for var in ["pc", "jetson"]:
    ok = G[(G["var"] == var) & (G.w_design == 0)].assign(zh=lambda d: d.z_hi.fillna(0))
    ok = ok.sort_values(["s_design", "z0", "g0", "a_low", "i_hi", "dz_wide", "zh"], ascending=[False, False, False, False, False, True, True])
    print(var, "rules with 0 wrong on all five NUDT sets:", len(ok)); display(ok.head(8).drop(columns=["zh"]).round(2))
    top = ok.iloc[0]
    CHOSEN[var] = dict(a_low=int(top.a_low), z0=float(top.z0), g0=float(top.g0), i_hi=int(top.i_hi), dz_wide=float(top.dz_wide), z_hi=None if pd.isna(top.z_hi) else float(top.z_hi))
CHOSEN

576 rules, 139 s
pc rules with 0 wrong on all five NUDT sets: 108


,var,a_low,z0,g0,i_hi,dz_wide,z_hi,s_design,w_design,s_nudt101,s_nudt202,s_nudt404,s_nudt505,s_nudt606
664,pc,5,2.0,0.0,12,0.0,NaN,56.18,0,55.12,56.46,56.4,56.64,56.28
666,pc,5,2.0,0.0,12,0.0,1.3,56.18,0,55.12,56.46,56.4,56.64,56.28
376,pc,4,2.0,0.0,12,0.0,NaN,56.18,0,55.12,56.46,56.4,56.64,56.28
378,pc,4,2.0,0.0,12,0.0,1.3,56.18,0,55.12,56.46,56.4,56.64,56.28
668,pc,5,2.0,0.0,12,0.3,NaN,56.12,0,55.12,56.16,56.4,56.64,56.28
670,pc,5,2.0,0.0,12,0.3,1.3,56.12,0,55.12,56.16,56.4,56.64,56.28
380,pc,4,2.0,0.0,12,0.3,NaN,56.12,0,55.12,56.16,56.4,56.64,56.28
382,pc,4,2.0,0.0,12,0.3,1.3,56.12,0,55.12,56.16,56.4,56.64,56.28


jetson rules with 0 wrong on all five NUDT sets: 164


,var,a_low,z0,g0,i_hi,dz_wide,z_hi,s_design,w_design,s_nudt101,s_nudt202,s_nudt404,s_nudt505,s_nudt606
657,jetson,5,2.0,0.0,10,0.0,NaN,55.51,0,54.92,55.26,56.7,55.64,55.02
659,jetson,5,2.0,0.0,10,0.0,1.3,55.51,0,54.92,55.26,56.7,55.64,55.02
661,jetson,5,2.0,0.0,10,0.3,NaN,55.51,0,54.92,55.26,56.7,55.64,55.02
663,jetson,5,2.0,0.0,10,0.3,1.3,55.51,0,54.92,55.26,56.7,55.64,55.02
729,jetson,5,2.2,0.0,10,0.0,NaN,55.39,0,54.72,55.26,56.7,55.24,55.02
731,jetson,5,2.2,0.0,10,0.0,1.3,55.39,0,54.72,55.26,56.7,55.24,55.02
705,jetson,5,2.0,0.5,10,0.0,NaN,55.37,0,54.92,54.96,56.7,55.24,55.02
707,jetson,5,2.0,0.5,10,0.0,1.3,55.37,0,54.92,54.96,56.7,55.24,55.02


{'pc': {'a_low': 5,
  'z0': 2.0,
  'g0': 0.0,
  'i_hi': 12,
  'dz_wide': 0.0,
  'z_hi': None},
 'jetson': {'a_low': 5,
  'z0': 2.0,
  'g0': 0.0,
  'i_hi': 10,
  'dz_wide': 0.0,
  'z_hi': None}}

**Read:** the same tier-2 test wins for both machines: **z >= 2.0 and zgap >= 0** (DINO's top place of its query), from **5 inliers**; tier 1 at
12 inliers on the PC (all 27 queries) and 10 on the Jetson (M = 5, cap 10). On the PC, a_low = 4 ties with 5 on NUDT (same score, 0 wrong); the
stricter 5 is the choice, and 4 is carried as a second finalist because it differs on official TRY (below). On the Jetson a_low = 4 makes 3 wrong
answers on NUDT.

## 2. Check on official TRY, with the plain-bar curve and where the gain comes from

In [3]:
FRONT = pd.read_csv(RES / "it2_bar_frontier.csv")
pc4 = {**CHOSEN["pc"], "a_low": 4}
runs = {("pc", "bar 8 / 11 (base)"): baseline(table, 8, 11, min_choice=3),
        ("pc", "two-tier"): decide_two_tier(table, CHOSEN["pc"], "pc"),
        ("pc", "two-tier, a_low 4"): decide_two_tier(table, pc4, "pc"),
        ("jetson", "Jetson today (base)"): baseline(table, 8, 8, jetson=True, m=5),
        ("jetson", "two-tier"): decide_two_tier(table, CHOSEN["jetson"], "jetson")}
rows = []
for (var, name), r in runs.items():
    base = runs[(var, "bar 8 / 11 (base)" if var == "pc" else "Jetson today (base)")]
    v = versus_frontier(r, FRONT, var).set_index("data"); a = answer_change(base, r).set_index("data")
    for d in SEEN:
        rows.append(dict(var=var, method=name, data=d, **v.loc[d].to_dict(), **a.loc[d, ["gain", "added_right", "added_wrong", "removed", "removed_wrong"]].to_dict()))
CHECK = pd.DataFrame(rows); CHECK.to_csv(RES / "it3_check_try.csv", index=False)
display(CHECK.set_index(["var", "method", "data"]).round(2))

score  wrong  n_answered  queries  same_n_bar  bar_score  bar_wrong  above_bar  gain  added_right  added_wrong  removed  removed_wrong
var    method              data                                                                                                                                                
pc     bar 8 / 11 (base)   official_try  51.37    0.0       162.0     8.95         8.0      51.37        0.0       0.00  0.00          0.0          0.0      0.0            0.0
                           nudt101       54.84    0.0       275.0     8.45         8.0      54.84        0.0       0.00  0.00          0.0          0.0      0.0            0.0
                           nudt202       57.06    0.0       192.0     7.47         8.0      57.06        0.0       0.00  0.00          0.0          0.0      0.0            0.0
                           nudt404       54.24    2.0       188.0     7.58         8.0      54.24        2.0       0.00  0.00          0.0          0.0      0.0            0.0
                           nudt505       54.24    1.0       275.0     8.38         8.0      54.24        1.0       0.00  0.00          0.0          0.0      0.0            0.0
                           nudt606       55.08    2.0       285.0     7.54         8.0      55.08        2.0       0.00  0.00          0.0          0.0      0.0            0.0
       two-tier            official_try  54.36    0.0       172.0     8.38         4.0      54.81        0.0      -0.45  2.98         10.0          0.0      0.0            0.0
                           nudt101       55.12    0.0       276.0     8.24         8.0      54.84        0.0       0.28  0.28          3.0          0.0      2.0            0.0
                           nudt202       56.46    0.0       190.0     7.89        10.0      56.46        0.0       0.00 -0.60          0.0          0.0      2.0            0.0
                           nudt404       56.40    0.0       190.0     7.68         7.0      53.52        4.0       2.88  2.16          3.0          0.0      1.0            1.0
                           nudt505       56.64    0.0       284.0     7.82         6.0      55.48        2.0       1.16  2.40         10.0          0.0      1.0            1.0
                           nudt606       56.28    0.0       285.0     7.75         8.0      55.08        2.0       1.20  1.20          3.0          0.0      3.0            2.0
       two-tier, a_low 4   official_try  56.17    0.0       178.0     8.04         3.0      55.20        1.0       0.97  4.80         16.0          0.0      0.0            0.0
                           nudt101       55.12    0.0       276.0     8.24         8.0      54.84        0.0       0.28  0.28          3.0          0.0      2.0            0.0
                           nudt202       56.46    0.0       190.0     7.89        10.0      56.46        0.0       0.00 -0.60          0.0          0.0      2.0            0.0
                           nudt404       56.40    0.0       190.0     7.68         7.0      53.52        4.0       2.88  2.16          3.0          0.0      1.0            1.0
                           nudt505       56.64    0.0       284.0     7.82         6.0      55.48        2.0       1.16  2.40         10.0          0.0      1.0            1.0
                           nudt606       56.28    0.0       285.0     7.68         8.0      55.08        2.0       1.20  1.20          3.0          0.0      3.0            2.0
jetson Jetson today (base) official_try  51.37    0.0       162.0     6.53         8.0      51.37        0.0       0.00  0.00          0.0          0.0      0.0            0.0
                           nudt101       54.24    0.0       272.0     6.43         8.0      54.24        0.0       0.00  0.00          0.0          0.0      0.0            0.0
                           nudt202       55.26    0.0       186.0     6.30         8.0      55.26        0.0       0.00  0.00          0.0          0.0      0.0         

## 3. Jetson early exit: answer a standard-query choice at once when it passes the rule with >= E inliers

In [4]:
rows = []
for E in [None, 12, 15, 20, 25, 30, 40]:
    s = summary(decide_two_tier(table, {**CHOSEN["jetson"], "early": E}, "jetson"))
    rows.append(dict(E=E, nudt_score=s.loc[NUDT_DESIGN, "score"].mean(), nudt_wrong=int(s.loc[NUDT_DESIGN, "wrong"].sum()), nudt_queries=s.loc[NUDT_DESIGN, "queries"].mean(),
                     try_score=s.loc["official_try", "score"], try_wrong=int(s.loc["official_try", "wrong"]), try_queries=s.loc["official_try", "queries"]))
EARLY = pd.DataFrame(rows); EARLY.to_csv(RES / "it3_early_exit.csv", index=False); display(EARLY.round(2))
ref = EARLY[EARLY.E.isna()].iloc[0]
ok = EARLY[EARLY.E.notna() & (EARLY.nudt_score >= ref.nudt_score - 0.1) & (EARLY.try_score >= ref.try_score - 0.1)]
E_CHOSEN = int(ok.E.min()); print("early-exit level: the lowest E within 0.1 of no early exit on NUDT and TRY ->", E_CHOSEN)

,E,nudt_score,nudt_wrong,nudt_queries,try_score,try_wrong,try_queries
0,NaN,55.51,0,6.32,52.86,0,6.42
1,12.0,55.43,0,2.04,52.12,0,2.74
2,15.0,55.42,0,2.15,52.74,0,3.03
3,20.0,55.44,0,2.38,52.86,0,3.58
4,25.0,55.46,0,2.67,52.86,0,4.07
5,30.0,55.45,0,2.94,52.74,0,4.53
6,40.0,55.45,0,3.49,52.74,0,5.20


early-exit level: the lowest E within 0.1 of no early exit on NUDT and TRY -> 20


The T7 early exit (judge P >= 0.98) cost ~1.2 points on official TRY because it answered before the most precise query. Requiring many
inliers for an early answer keeps the precise answers (the extra queries mostly matter for weak photos).

## 4. Attitude correction (T6), fitted on TRY only, cross-checked by place inside TRY

In [5]:
DUMP = str(ROOT / "tests/Test_Iteration_6/results/rich_dump/official")
b0 = runs[("pc", "bar 8 / 11 (base)")]; fr = attitude_frame(b0[b0.data == "official_try"], DUMP)
fr = fr[fr.data == "official_try"].merge(pd.read_csv(RES / "official_split.csv")[["photo", "block"]], on="photo")
pts = lambda e: np.select([e <= 1, e <= 2, e <= 4, e <= 8], [10, 6, 3, 0], -20)
rows = []
for terms in [("angle",), ("angle", "const"), ("angle", "tilt", "const")]:
    new = np.full(len(fr), np.nan)
    for b in fr.block.unique():
        tr, te = (fr.block != b).to_numpy(), (fr.block == b).to_numpy()
        ex, ey = pose_bias.apply(fr[te], pose_bias.fit(fr[tr], terms)); new[te] = np.hypot(ex, ey) / fr.sigma[te]
    k = (fr.err_sigma <= 8).to_numpy()
    rows.append(dict(terms="+".join(terms), n=int(k.sum()), pts_before=pts(fr.err_sigma[k]).mean(), pts_after=pts(new[k]).mean(),
                     within1_before=(fr.err_sigma[k] <= 1).mean(), within1_after=(new[k] <= 1).mean()))
display(pd.DataFrame(rows).round(3))
ATT = pose_bias.fit(fr, ("angle",))
print("fit on all TRY answers (terms: angle): forward %.3f deg, lateral %.3f deg, mirror %s" % (np.degrees(ATT["coef"][0]), np.degrees(ATT["coef"][1]), ATT["mirror"]))

,terms,n,pts_before,pts_after,within1_before,within1_after
0,angle,162,9.778,9.951,0.944,0.988
1,angle+const,162,9.778,9.975,0.944,0.994
2,angle+tilt+const,162,9.778,9.926,0.944,0.981


fit on all TRY answers (terms: angle): forward 1.033 deg, lateral 0.049 deg, mirror False


**Read:** held out by place inside TRY, a constant camera angle (2 numbers, ~1.0 deg forward; T6 found 0.98 deg on all official photos) moves
every answer within 1 sigma. The simplest model (angle only) is as good as the larger ones: it is the finalist. It needs the official camera's
attitude, so it cannot be checked on NUDT.

## 5. Pre-registration

In [6]:
pc_tau = max(ltt_tau(scan_tau(table, judge_probability(table, NUDT_DESIGN, FEATURES_CTX, "gbm")[0], TAUS, override=12, min_choice=3, wide_tau=0.95), NUDT_DESIGN, 0.005), 0.9)
pj = judge_probability(table, NUDT_DESIGN, FEATURES_CTX, "gbm")[0]
jet_tau = max(ltt_tau(scan_tau(table, pj, TAUS, jetson=True, m=5, override=12, min_choice=3, wide_tau=0.95), NUDT_DESIGN, 0.005), 0.9)
PRE = dict(
    written=datetime.datetime.now().isoformat(timespec="seconds"),
    final_sets=["official_test", "nudt707"],
    note="Scored once in It4. official_test: 194 official photos in places never used to build or check anything here (split by place). "
         "nudt707: 300 fresh NUDT photos (seed 707, pool 2400, the 500-pool and every 404/505/606 photo excluded), dumped before any T8 rule.",
    disclosure="During exploration one printout showed official_test scores of three NUDT-only judge variants (feature sets) before the test half was "
               "locked; no choice in T8 used it. Every earlier method (T6/T7) was designed while looking at all 379 official photos.",
    pc=dict(base=dict(bars=[8, 11]),
            two_tier=CHOSEN["pc"], two_tier_a4=pc4,
            judge_nudt_only=dict(features="FEATURES_CTX", kind="gbm", trained_on=NUDT_DESIGN, tau=pc_tau, override=12, min_choice=3, wide_tau=0.95)),
    jetson=dict(base=dict(bar=8, cap=10, m=5),
                two_tier=CHOSEN["jetson"], two_tier_early={**CHOSEN["jetson"], "early": E_CHOSEN},
                judge_nudt_only=dict(features="FEATURES_CTX", kind="gbm", trained_on=NUDT_DESIGN, tau=jet_tau, override=12, min_choice=3, wide_tau=0.95),
                judge_nudt_only_early=dict(early_tau=0.98)),
    attitude=dict(terms=list(ATT["terms"]), coef=[float(c) for c in ATT["coef"]], mirror=bool(ATT["mirror"]), fitted_on="official_try answers of bar 8 / 11",
                  applies_to="official answers only (needs the official camera attitude)"),
    report=["score /60, wrong answers, answered, queries per photo", "paired bootstrap gain over the base of the same machine (95 %)",
            "the plain bar answering as many photos (versus_frontier)", "answer_change: added right / wrong, removed"],
    success="a finalist counts as a gain if on BOTH official_test and nudt707 it has 0 wrong answers (or no more than its base) and a gain >= 0, "
            "and on at least one of them a paired-bootstrap interval above 0")
path = RES / "preregistered_t8.json"
assert not path.exists(), "pre-registration already written: do not overwrite"
path.write_text(json.dumps(PRE, indent=1))
print(path.read_text()); print("sha256", hashlib.sha256(path.read_bytes()).hexdigest())

{
 "written": "2026-11-14T01:47:03",
 "final_sets": [
  "official_test",
  "nudt707"
 ],
 "note": "Scored once in It4. official_test: 194 official photos in places never used to build or check anything here (split by place). nudt707: 300 fresh NUDT photos (seed 707, pool 2400, the 500-pool and every 404/505/606 photo excluded), dumped before any T8 rule.",
 "disclosure": "During exploration one printout showed official_test scores of three NUDT-only judge variants (feature sets) before the test half was locked; no choice in T8 used it. Every earlier method (T6/T7) was designed while looking at all 379 official photos.",
 "pc": {
  "base": {
   "bars": [
    8,
    11
   ]
  },
  "two_tier": {
   "a_low": 5,
   "z0": 2.0,
   "g0": 0.0,
   "i_hi": 12,
   "dz_wide": 0.0,
   "z_hi": null
  },
  "two_tier_a4": {
   "a_low": 4,
   "z0": 2.0,
   "g0": 0.0,
   "i_hi": 12,
   "dz_wide": 0.0,
   "z_hi": null
  },
  "judge_nudt_only": {
   "features": "FEATURES_CTX",
   "kind": "gbm",
   "trained